In [ ]:
import geopandas as gpd
import pandas as pd
import fiona
import numpy as np
import os
import time
from shapely import wkb
from shapely.geometry import Point
import polars as pl
# Enable KML driver for KMZ reading
fiona.drvsupport.supported_drivers['KML'] = 'rw'
fiona.drvsupport.supported_drivers['LIBKML'] = 'rw'


In [ ]:
os.chdir('..')
os.getcwd()

In [ ]:
backbone = gpd.read_parquet('data/processed/backbone_roads.parquet')
backbone

In [ ]:
gas_stations_proximity = gpd.read_parquet('data/processed/gas_stations_proximity.parquet') 
gas_stations_proximity

In [ ]:
segments = segments.join(charging_metrics,how='inner',on='segment_id')
segments

In [ ]:
ev_range = 100000
pct_ev_traffic = 0.2

In [ ]:
segments = segments.with_columns(
    (pl.col('avg_gap_m')/ev_range).alias('charge_prob')
)
segments

In [ ]:
segments.select('avg_gap_m','charge_prob').describe()

In [ ]:
segments = segments.with_columns(
    (pl.col('total_max')*pct_ev_traffic*pl.col('pct_not_short_max')*pl.col('charge_prob')).alias('demand')
)
segments

In [ ]:
segments.select('avg_gap_m','charge_prob','demand').describe()

In [ ]:
output_path

In [ ]:
segments = pd.read_parquet(output_path)
segments

In [ ]:
segments.sort_values(by='avg_gap_m',ascending=False)[80:100]

In [ ]:
segments.describe()

In [ ]:
target_crs = "EPSG:3042"
gdf_segments = gpd.read_parquet(integrated_road_path)
print(f"Loaded {len(gdf_segments)} road segments. CRS: {gdf_segments.crs}")
gdf_segments.head()

In [ ]:
gdf_stations = gpd.read_parquet(charging_stations_path)
print(f"Loaded {len(gdf_stations)} charging stations. CRS: {gdf_stations.crs}")

In [ ]:
gdf_stations.head()

In [ ]:
gdf_all_backbones = gpd.read_file(kmz_path, driver='KML')
gdf_all_backbones['backbone_id'] = gdf_all_backbones.index.astype(float)
print(f"Loaded {len(gdf_all_backbones)} backbone roads")
gdf_all_backbones.head()

In [ ]:
# 2. Filter Backbones to match those in integrated road network
relevant_backbone_ids = gdf_segments['backbone_id'].unique()
gdf_backbone = gdf_all_backbones[gdf_all_backbones['backbone_id'].isin(relevant_backbone_ids)].copy()
print(f"Filtered to {len(gdf_backbone)} relevant backbones.")

In [ ]:
gdf_segments = gdf_segments.to_crs(target_crs)
gdf_stations = gdf_stations.to_crs(target_crs)
gdf_backbone = gdf_backbone.to_crs(target_crs)

In [ ]:
results = []
    
# 4. Processing Loop by Backbone
backbone_groups = gdf_segments.groupby('backbone_id')
total_backbones = len(backbone_groups)

In [ ]:
start_time = time.time()
for i, (b_id, segments) in enumerate(backbone_groups):
    if i % 100 == 0:
            print(f"Processing backbone {i}/{total_backbones}... (Elapsed: {time.time()-start_time:.1f}s)")

In [ ]:
gdf_segments.head()

In [ ]:
segments = gdf_segments.groupby('backbone_id').agg({'segment_id':'count','original_segment_count':'sum'}).reset_index()
segments

In [ ]:
gdf_backbone['length_m'] = gdf_backbone.geometry.length

In [ ]:
gdf_backbone.sort_values(by='length_m',ascending=False)

In [ ]:
b_id = 1377.0

In [ ]:
segments = gdf_segments.loc[gdf_segments['backbone_id']==b_id]
segments

In [ ]:
backbone_row = gdf_backbone[gdf_backbone['backbone_id'] == b_id]
backbone_row

In [ ]:
backbone_geom = backbone_row.geometry.iloc[0]

In [ ]:
backbone_row_plot = backbone_row.to_crs(4326)
backbone_geom_plot = backbone_row_plot.geometry.iloc[0]

In [ ]:
local_stations = gdf_stations[gdf_stations['backbone_id'] == b_id].copy()
local_stations

In [ ]:
local_stations['mile_marker'] = local_stations.geometry.apply(lambda p: backbone_geom.project(p))
local_stations = local_stations.sort_values('mile_marker')

In [ ]:
local_stations

In [ ]:
import folium
from folium.plugins import MarkerCluster

In [ ]:
# Individual Points Visualization (No Clustering)
m_roads_charging_points = folium.Map(location=[40.4168, -3.7038], zoom_start=12, tiles='cartodbpositron')

for row in local_stations.itertuples():
    folium.CircleMarker(
        location=[row.latitude, row.longitude],
        radius=3,
        color="#3186cc",
        fill=True,
        fill_color="#3186cc",
        fill_opacity=0.7,
        tooltip=getattr(row, "site_name", "")
    ).add_to(m_roads_charging_points)


# Add Processed Segments
folium.GeoJson(
    backbone_geom_plot,
    name="Processed Network",
    style_function=lambda x: {'color': 'blue', 'weight': 2, 'opacity': 0.7},
    #tooltip=folium.GeoJsonTooltip(fields=["backbone_id", "original_segment_count"], aliases=["Backbone ID:", "Original Segments:"])
).add_to(m_roads_charging_points)

m_roads_charging_points

In [ ]:
for _, seg_row in segments.iterrows():
    seg_geom = seg_row.geometry

In [ ]:
seg_geom

In [ ]:
if seg_geom.geom_type == 'Point':
    print('point')
    span = [backbone_geom.project(seg_geom)]
elif seg_geom.geom_type.startswith('Multi'):
    print('Multi')
    span = [backbone_geom.project(Point(p)) for part in seg_geom.geoms for p in part.coords]
else:
    print('project to point ')
    span = [backbone_geom.project(Point(p)) for p in seg_geom.coords]

In [ ]:
span =[backbone_geom.project(Point(p)) for part in seg_geom.geoms for p in part.coords]

In [ ]:
s_min, s_max = min(span), max(span)
print(s_min,s_max)

In [ ]:
inside = local_stations[(local_stations['mile_marker'] >= s_min) & (local_stations['mile_marker'] <= s_max)]
inside

In [ ]:
behind = local_stations[local_stations['mile_marker'] < s_min].tail(1)
behind

In [ ]:
ahead = local_stations[local_stations['mile_marker'] > s_max].head(1)
ahead      

In [ ]:
local_set = pd.concat([inside, behind, ahead]).drop_duplicates(subset=['site_id']).sort_values('mile_marker')
local_set

In [ ]:
#create_result_entry(seg_row, b_id, local_set)

In [ ]:
seg_row

In [ ]:
b_id

In [ ]:
local_stations = local_set

In [ ]:
entry = {
        'segment_id': seg_row['segment_id'],
        'backbone_id': b_id,
        'num_charging_stations': 0,
        'num_chargers': 0,
        'avg_gap_m': np.nan,
        'max_gap_m': np.nan,
        'has_charger_inside': False
    }

In [ ]:
not isinstance(local_stations, list) and not local_stations.empty

In [ ]:
entry['num_charging_stations'] = len(local_stations)
len(local_stations)

In [ ]:
entry['num_chargers'] = local_stations['n_chargers'].sum()
local_stations['n_chargers'].sum()

In [ ]:
len(local_stations) >= 2

In [ ]:
local_stations['mile_marker']

In [ ]:
gaps = np.diff(local_stations['mile_marker'].values)
gaps

In [ ]:
entry['avg_gap_m'] = np.mean(gaps)
np.mean(gaps)

In [ ]:
entry['max_gap_m'] = np.max(gaps)
entry['max_gap_m']

In [ ]:
entry

In [ ]:
pd.read_parquet('data/processed/road_segment_charging_metrics.parquet')